<a href="https://colab.research.google.com/github/anbaluxe/my_colab_learning/blob/main/%D0%A1%D1%82%D1%80%D0%B0%D1%82%D0%B8%D1%84%D0%B8%D1%86%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%BD%D0%B4%D0%BE%D0%BC%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_users = 10_000

df = pd.DataFrame({
    "user_id": np.arange(1, n_users + 1),
    "os": np.random.choice(
        ["Android", "iOS"],
        size=n_users,
        p=[0.7, 0.3]
    )
})

df.head()

,user_id,os
0,1,Android
1,2,iOS
2,3,iOS
3,4,Android
4,5,Android


In [27]:
df["os"].value_counts(normalize=True)

,proportion
os,
Android,0.7113
iOS,0.2887


In [28]:
df["rpu"] = np.where(
    df["os"] == "Android",
    np.random.normal(20, 5, n_users),
    np.random.normal(100, 10, n_users)
)

In [29]:
df.groupby("os")["rpu"].agg(["count", "mean", "var", "std"])

,count,mean,var,std
os,,,,
Android,7113,20.123618,25.252115,5.025148
iOS,2887,99.992981,98.242981,9.911760


## Рандомизация по OS

In [30]:
df['group'] = 'Control' # Заполняем все строки Control

test_idx = (
    df.groupby('os')
      .sample(frac=0.5, random_state=42)
      .index
)

'''
  1. Разделяем по os
  2. Случайная рандомизация внутри каждой группы frac=0.5 50%
  3. Получаемя index-сы пользоваателей
'''

df.loc[test_idx, 'group'] = 'Test'

In [31]:
pd.crosstab(
    df["os"],
    df["group"]
)

group,Control,Test
os,,
Android,3557,3556
iOS,1443,1444


In [32]:
pd.crosstab(
    df["os"],
    df["group"],
    normalize="index"
)

group,Control,Test
os,,
Android,0.500070,0.499930
iOS,0.499827,0.500173


In [33]:
pd.crosstab(
    df["group"],
    df["os"],
    normalize="index"
)

os,Android,iOS
group,,
Control,0.7114,0.2886
Test,0.7112,0.2888


In [34]:
df["rpu_test"] = df["rpu"]

df.loc[df["group"] == "Test", "rpu_test"] *= 1.05

## Анализ стратифицированной рандомизации

In [35]:
def analyze_ab(data, metric_col="rpu_test"):
    stats = data.groupby("group")[metric_col].agg(
        ["count", "mean", "var", "std"]
    )

    stats["var_mean"] = stats["var"] / stats["count"]

    se = (
        stats.loc["Test", "var_mean"]
        + stats.loc["Control", "var_mean"]
    ) ** 0.5

    delta = stats.loc["Test", "mean"] - stats.loc["Control", "mean"]
    z_stat = delta / se
    ci_left = delta - 1.96 * se
    ci_right = delta + 1.96 * se

    return stats, delta, se, z_stat, (ci_left, ci_right)

In [36]:
strat, delta_strat, se_strat, z_stat_strat, ci_strat = analyze_ab(df)

print(strat)
print("SE", se_strat)
print("delta", delta_strat)
print("z-stat", z_stat_strat)
print("95% CI", ci_strat)

         count       mean          var        std  var_mean
group                                                      
Control   5000  43.043866  1342.557604  36.640928  0.268512
Test      5000  45.485938  1510.979975  38.871326  0.302196
SE 0.7554518619672222
delta 2.4420714363207097
z-stat 3.2325970181097614
95% CI (np.float64(0.961385786864954), np.float64(3.9227570857764653))


## Без стратификации

In [37]:
df_unstrat = df[["user_id", "os", "rpu"]].copy()

df_unstrat["group"] = "Control"

test_idx = (
    df_unstrat
    .sample(frac=0.5, random_state=42)
    .index
)

df_unstrat.loc[test_idx, "group"] = "Test"

df_unstrat["rpu_test"] = df_unstrat["rpu"]
df_unstrat.loc[df_unstrat["group"] == "Test", "rpu_test"] *= 1.05

In [38]:
pd.crosstab(
    df_unstrat["group"],
    df_unstrat["os"]
)

os,Android,iOS
group,,
Control,3535,1465
Test,3578,1422


In [39]:
pd.crosstab(
    df_unstrat["group"],
    df_unstrat["os"],
    normalize="index"
)

os,Android,iOS
group,,
Control,0.7070,0.2930
Test,0.7156,0.2844


## Анализ обычной рандомизации

In [40]:
unstrat, delta_unstrat, se_unstrat, z_stat_unstrat, ci_unstrat = analyze_ab(df_unstrat)

print(unstrat)
print("SE", se_unstrat)
print("delta", delta_unstrat)
print("z-stat", z_stat_unstrat)
print("95% CI", ci_unstrat)

         count       mean          var        std  var_mean
group                                                      
Control   5000  43.580847  1369.300774  37.004064  0.273860
Test      5000  44.922108  1481.186645  38.486188  0.296237
SE 0.7550480009069714
delta 1.3412614999018118
z-stat 1.776392359546247
95% CI (np.float64(-0.13863258187585226), np.float64(2.8211555816794758))


## Сравнение Страт.Рандомизации и Обычной

In [41]:
comparison = pd.DataFrame({
    "Stratified randomization": [delta_strat, se_strat, z_stat_strat, *ci_strat],
    "Unstratified randomization": [delta_unstrat, se_unstrat, z_stat_unstrat, *ci_unstrat]
}, index=["delta", "SE", "z-stat", "CI left", "CI right"])

comparison

,Stratified randomization,Unstratified randomization
delta,2.442071,1.341261
SE,0.755452,0.755048
z-stat,3.232597,1.776392
CI left,0.961386,-0.138633
CI right,3.922757,2.821156


## Пост-стратификация

In [42]:
post_df = (
    df_unstrat.groupby(['group', 'os'])['rpu']
      .agg(['count', 'var'])
      .rename(columns={
          'count': 'n',
          'var': 'variance'
      })
)

post_df

n   variance
group   os                      
Control Android  3535  25.980202
        iOS      1465  97.300190
Test    Android  3578  24.535123
        iOS      1422  99.272827

1. Высчитываем вес

In [43]:
weights = df['os'].value_counts(normalize=True)

post_df['weight'] = post_df.index.get_level_values('os').map(weights)

2. Рассчитаем дисперсию стратифицированного среднего

In [44]:
post_df['variance_contribution'] = (
    post_df['weight'] ** 2
    * post_df['variance']
    / post_df['n']
)

var_strats = post_df.groupby(level='group')['variance_contribution'].sum()
var_strats

,variance_contribution
group,
Control,0.009254
Test,0.009288


3. Считаем среднее значение

In [45]:
means = df.groupby(['group', 'os'])['rpu'].mean()

post_df['mean'] = post_df.index.map(means)

post_df

n   variance  weight  variance_contribution        mean
group   os                                                                 
Control Android  3535  25.980202  0.7113               0.003718   20.119246
        iOS      1465  97.300190  0.2887               0.005536   99.553135
Test    Android  3578  24.535123  0.7113               0.003469   20.127993
        iOS      1422  99.272827  0.2887               0.005819  100.432522

4. Считаем среднее с учетом веса страты

In [46]:
post_df['weight_mean'] = post_df['weight'] * post_df['mean']

post_df

n   variance  weight  variance_contribution        mean  \
group   os                                                                    
Control Android  3535  25.980202  0.7113               0.003718   20.119246   
        iOS      1465  97.300190  0.2887               0.005536   99.553135   
Test    Android  3578  24.535123  0.7113               0.003469   20.127993   
        iOS      1422  99.272827  0.2887               0.005819  100.432522   

                 weight_mean  
group   os                    
Control Android    14.310819  
        iOS        28.740990  
Test    Android    14.317041  
        iOS        28.994869

5. Считаем стратифицированное среднее по группам

In [47]:
post_means = post_df.groupby(level='group')['weight_mean'].sum()

6. Считаем delta и se

In [48]:
delta_post = post_means['Test'] - post_means['Control']
se_post = (var_strats['Test'] + var_strats['Control']) ** 0.5

In [49]:
print('Delta: ', delta_post)
print('SE: ', se_post)

Delta:  0.26010056579134755
SE:  0.1361695885190418


In [50]:
z_stat = delta_post / se_post
print(z_stat)

1.9101222866291867


In [51]:
post_ci_left = delta_post - 1.96 * se_post
post_ci_right = delta_post + 1.96 * se_post
ci_post = (post_ci_left, post_ci_right)
print(ci_post)

(np.float64(-0.006791827705974363), np.float64(0.5269929592886695))


## Сравнение Стратификации, Обычное рандомизации, Пост-Стратификации

In [52]:
comparison = pd.DataFrame({
    "Stratified randomization": [delta_strat, se_strat, z_stat_strat, *ci_strat],
    "Unstratified randomization": [delta_unstrat, se_unstrat, z_stat_unstrat, *ci_unstrat],
    'PostStratified randomization': [delta_post, se_post, z_stat, *ci_post]
}, index=["delta", "SE", "z-stat", "CI left", "CI right"])

comparison

,Stratified randomization,Unstratified randomization,PostStratified randomization
delta,2.442071,1.341261,0.260101
SE,0.755452,0.755048,0.136170
z-stat,3.232597,1.776392,1.910122
CI left,0.961386,-0.138633,-0.006792
CI right,3.922757,2.821156,0.526993
